# Pharmacokinetics Grapher

Visualize medication concentration curves using one-compartment pharmacokinetic modeling.

> **Educational use only.** Approximate relative concentration curves based on simplified PK models. Not for medical dosing decisions.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pk_core import (
    Prescription, DoseStep, FREQUENCY_MAP, DEFAULT_TIMES,
    calculate_concentration, accumulate_doses, accumulate_metabolite_doses,
    accumulate_schedule, calculate_milestones, compute_steady_state_metrics,
    save_prescriptions, load_prescriptions, generate_frequency_variants,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

---
## Phase 1: Single-Dose Visualization

In [ ]:
# Example: Ibuprofen 400mg
ibuprofen = Prescription(
    name='Ibuprofen',
    dose=400,
    half_life=2.0,
    uptake=0.5,
    peak=1.5,
    frequency='tid',
    times=['08:00', '14:00', '20:00'],
)

t = np.linspace(0, 12, 500)
c = calculate_concentration(t, ibuprofen.dose, ibuprofen.half_life, ibuprofen.uptake)
c_normalized = c / c.max() if c.max() > 0 else c

fig, ax = plt.subplots()
ax.plot(t, c_normalized, linewidth=2)
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Relative Concentration (peak = 1.0)')
ax.set_title(f'{ibuprofen.name} — Single Dose ({ibuprofen.dose}mg)')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Phase 2: Multi-Dose Accumulation

In [ ]:
# Ibuprofen tid over 48 hours — observe accumulation
t, c = accumulate_doses(ibuprofen, start_hours=0, end_hours=48)

fig, ax = plt.subplots()
ax.plot(t, c, linewidth=2, label=f'{ibuprofen.name} {ibuprofen.dose}mg ({ibuprofen.frequency})')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Relative Concentration (peak = 1.0)')
ax.set_title('Multi-Dose Accumulation')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Multi-Drug Comparison

In [ ]:
def plot_prescriptions(prescriptions, end_hours=48, title='PK Comparison'):
    """Overlay multiple drug curves on one plot."""
    fig, ax = plt.subplots()
    colors = plt.cm.tab10.colors

    for i, rx in enumerate(prescriptions):
        color = colors[i % len(colors)]

        t, c = accumulate_doses(rx, end_hours=end_hours)
        label = f'{rx.name} {rx.dose}mg ({rx.frequency})'
        ax.plot(t, c, linewidth=2, color=color, label=label)

        met = accumulate_metabolite_doses(rx, end_hours=end_hours)
        if met is not None:
            t_m, c_m = met
            met_label = f'{rx.name} — Metabolite'
            ax.plot(t_m, c_m, linewidth=1.5, color=color, linestyle='--', label=met_label)

    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Relative Concentration (peak = 1.0)')
    ax.set_title(title)
    ax.set_ylim(0)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


acetaminophen = Prescription(
    name='Acetaminophen',
    dose=500,
    half_life=3.0,
    uptake=0.75,
    peak=1.0,
    frequency='q6h',
    times=['06:00', '12:00', '18:00', '00:00'],
)

plot_prescriptions([ibuprofen, acetaminophen], end_hours=48, title='Ibuprofen vs Acetaminophen')

### PK Milestone Timeline

In [ ]:
def print_milestones(rx, end_hours=None):
    """Display milestone timeline as formatted table."""
    events = calculate_milestones(rx, end_hours=end_hours)

    print(f'\n  PK Timeline: {rx.name} {rx.dose}mg ({rx.frequency})')
    print(f'  {"─" * 65}')
    print(f'  {"Time (h)":>10}  {"Event":<12}  {"Level":>7}  {"Description"}')
    print(f'  {"─" * 65}')

    for e in events:
        time_str = f'{e["time_hours"]:>10.1f}'
        level_str = f'{e["level"]:>6.1f}%' if e['level'] is not None else '     —'
        print(f'  {time_str}  {e["event"]:<12}  {level_str}  {e["description"]}')


print_milestones(ibuprofen, end_hours=24)

---
## Phase 3: Analysis

### Steady-State Analysis

In [ ]:
def display_steady_state(rx):
    """Compute and display steady-state metrics."""
    m = compute_steady_state_metrics(rx)

    print(f'\n  Steady-State Analysis: {rx.name}')
    print(f'  {"─" * 45}')
    print(f'  Dosing interval (tau):   {m["tau"]:.1f} hours')
    print(f'  Elimination half-life:   {rx.half_life:.1f} hours')
    print(f'  Accumulation factor:     {m["accum_factor"]:.2f}x')
    print(f'  Time to steady-state:    ~{m["t_ss"]:.0f} hours ({m["t_ss"]/24:.1f} days)')
    print(f'  SS peak (normalized):    {m["ss_peak"]:.3f}')
    print(f'  SS trough (normalized):  {m["ss_trough"]:.3f}')
    print(f'  Peak-trough swing:       {m["swing"]:.3f}')
    return m


display_steady_state(ibuprofen);

### Parameter Sensitivity

In [ ]:
def sensitivity_plot(rx, param_name, values, end_hours=48):
    """Plot family of curves varying one parameter."""
    fig, ax = plt.subplots()
    cmap = plt.cm.viridis

    for i, val in enumerate(values):
        modified = Prescription(
            name=rx.name, dose=rx.dose, half_life=rx.half_life,
            uptake=rx.uptake, peak=rx.peak, frequency=rx.frequency,
            times=list(rx.times),
        )
        setattr(modified, param_name, val)

        t, c = accumulate_doses(modified, end_hours=end_hours)
        color = cmap(i / max(len(values) - 1, 1))
        ax.plot(t, c, linewidth=1.5, color=color, label=f'{param_name}={val}')

    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Relative Concentration')
    ax.set_title(f'Sensitivity: {rx.name} — varying {param_name}')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


sensitivity_plot(ibuprofen, 'half_life', [1.0, 1.5, 2.0, 3.0, 4.0], end_hours=48)

### Dosing Frequency Comparison

In [ ]:
variants = generate_frequency_variants(ibuprofen, ['bid', 'tid', 'qid', 'q6h'])
plot_prescriptions(variants, end_hours=48,
                   title=f'{ibuprofen.name} — Same Daily Dose, Different Frequencies')

### Titration / Taper Schedule

In [ ]:
prednisone = Prescription(
    name='Prednisone', dose=40, half_life=3.5, uptake=1.0, peak=2.0,
    frequency='qd', times=['08:00'],
)

taper_steps = [
    DoseStep(dose=40, duration_days=5),
    DoseStep(dose=30, duration_days=5),
    DoseStep(dose=20, duration_days=5),
    DoseStep(dose=10, duration_days=5),
    DoseStep(dose=5, duration_days=5),
]

t, c = accumulate_schedule(prednisone, taper_steps)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t, c, linewidth=2)
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Relative Concentration')
ax.set_title('Prednisone Taper Schedule')

day_offset = 0
for step in taper_steps:
    ax.axvline(x=day_offset * 24, color='gray', linestyle=':', alpha=0.5)
    ax.text(day_offset * 24 + 12, 0.95, f'{step.dose}mg',
            ha='center', fontsize=8, color='gray')
    day_offset += step.duration_days

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Your Prescriptions

Edit the cell below to define your own prescriptions, then run the plotting cells above.

In [ ]:
my_prescriptions = [
    Prescription(
        name='Drug A',
        dose=500,
        half_life=6.0,
        uptake=1.5,
        peak=2.0,
        frequency='bid',
        times=['09:00', '21:00'],
    ),
]

plot_prescriptions(my_prescriptions, end_hours=72)

for rx in my_prescriptions:
    print_milestones(rx, end_hours=48)

for rx in my_prescriptions:
    display_steady_state(rx)

---
## Load from Web App Export

Import prescriptions exported from the Pharmacokinetics Grapher web application.

In [ ]:
# Uncomment and edit the path to load from a JSON export:
# imported = load_prescriptions('my_export.json')
# plot_prescriptions(imported, end_hours=72)